In [1]:
!pip install pandas requests -q

import requests
import time
import pandas as pd
import numpy as np
from datetime import datetime

CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'ARBUSDT', 'BNBUSDT'],
    'INTERVAL': '5m',
    'START_DATE': '01/07/2023',
    'END_DATE': '01/01/2024'
}

class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                base_url = "https://api.binance.us"
                if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
                if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"
                if req_symbol == "SOLUSDT": req_symbol = "SOLUSD"
                if req_symbol == "ARBUSDT": req_symbol = "ARBUSD"
        except Exception:
            base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
        end_time = int(end_dt.timestamp() * 1000)

        all_klines = []
        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        return df

class ORBStatistician:
    def __init__(self, df):
        self.df = df.copy()

    def run_analysis(self):
        self.df['Date'] = self.df.index.date
        self.df['Hour'] = self.df.index.hour
        self.df['Minute'] = self.df.index.minute

        daily_close = self.df.groupby('Date')['Close'].last()
        self.df['Daily_Close_Price'] = self.df['Date'].map(daily_close)

        self.df['Return'] = (self.df['Daily_Close_Price'] - self.df['Close']) / self.df['Close']
        self.df['Is_Win'] = (self.df['Return'] > 0).astype(int)

        hourly_volume = self.df.groupby('Hour')['Volume'].sum()
        top_3_hours = hourly_volume.nlargest(3).index.tolist()

        filtered_df = self.df[self.df['Hour'].isin(top_3_hours)]

        stats = filtered_df.groupby(['Hour', 'Minute']).agg(
            Total_Days=('Is_Win', 'count'),
            Wins=('Is_Win', 'sum'),
            Avg_Return=('Return', 'mean')
        ).reset_index()

        stats['Time'] = stats['Hour'].astype(str).str.zfill(2) + ":" + stats['Minute'].astype(str).str.zfill(2)
        stats['Winrate_%'] = (stats['Wins'] / stats['Total_Days']) * 100
        stats['Avg_Return_%'] = stats['Avg_Return'] * 100

        stats = stats.sort_values(by='Winrate_%', ascending=False)
        stats = stats[['Time', 'Total_Days', 'Winrate_%', 'Avg_Return_%']].reset_index(drop=True)

        return top_3_hours, stats

if __name__ == "__main__":
    print("STARTING STATISTICAL ANALYSIS (M5 to Daily Close)\n")

    for symbol in CONFIG['SYMBOLS']:
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            analyzer = ORBStatistician(df_raw)
            top_hours, stats_df = analyzer.run_analysis()

            top_hours_str = ", ".join([f"{h:02d}:00 UTC" for h in top_hours])

            # Format numbers for clean display
            stats_df['Winrate_%'] = stats_df['Winrate_%'].map("{:.2f}".format)
            stats_df['Avg_Return_%'] = stats_df['Avg_Return_%'].map("{:.3f}".format)

            print("="*60)
            print(f" ASSET: {symbol}")
            print(f" Top 3 Volume Hours: {top_hours_str}")
            print("-" * 60)
            print(" TOP 5 BEST M5 ENTRY TIMES (Ranked by Winrate)")
            print("-" * 60)
            print(stats_df.head(5).to_string(index=False))
            print("="*60 + "\n")

STARTING STATISTICAL ANALYSIS (M5 to Daily Close)

 ASSET: BTCUSDT
 Top 3 Volume Hours: 15:00 UTC, 14:00 UTC, 16:00 UTC
------------------------------------------------------------
 TOP 5 BEST M5 ENTRY TIMES (Ranked by Winrate)
------------------------------------------------------------
 Time  Total_Days Winrate_% Avg_Return_%
16:55         185     59.46        0.186
16:50         185     57.30        0.185
15:25         185     57.30        0.169
15:50         185     57.30        0.187
15:30         185     56.22        0.167

 ASSET: ETHUSDT
 Top 3 Volume Hours: 15:00 UTC, 16:00 UTC, 14:00 UTC
------------------------------------------------------------
 TOP 5 BEST M5 ENTRY TIMES (Ranked by Winrate)
------------------------------------------------------------
 Time  Total_Days Winrate_% Avg_Return_%
16:55         185     56.22        0.134
16:50         185     55.14        0.138
16:45         185     54.59        0.125
16:25         185     54.59        0.097
16:30         185    

In [2]:
!pip install pandas requests plotly -q

import requests
import time
import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# ==========================================
# CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'ARBUSDT', 'BNBUSDT'],
    'INTERVAL': '5m',
    'START_DATE': '01/01/2023',
    'END_DATE': '01/01/2024',
    'OR_DURATION_MINS': 15,       # Thời gian hình thành hộp Open Range (15 phút)
    'SHOW_EDA_PLOTS': True        # Bật/Tắt hiển thị biểu đồ phân tích Volume
}

# ==========================================
# DATA FETCHER
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                base_url = "https://api.binance.us"
                if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
                if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"
                if req_symbol == "SOLUSDT": req_symbol = "SOLUSD"
                if req_symbol == "ARBUSDT": req_symbol = "ARBUSD"
        except: base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)
        end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
        end_time = int(end_dt.timestamp() * 1000)

        all_klines = []
        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        return df.astype(float)

# ==========================================
# ORB ANALYZER & EDA
# ==========================================
class TrueORBAnalyzer:
    def __init__(self, df, or_mins):
        self.df = df
        self.or_candles = int(or_mins / 5)

        # Dùng 'mean' (Trung bình) thay vì 'sum' để EDA chuẩn xác hơn
        # (Tránh việc các tháng có số ngày khác nhau làm lệch Data)
        self.hourly_vol = self.df.groupby(self.df.index.hour)['Volume'].mean()
        self.top_3_hours = self.hourly_vol.nlargest(3).index.tolist()

    def plot_volume_eda(self, symbol):
        """Vẽ biểu đồ phân phối Volume theo giờ để trực quan hóa Top 3 khung giờ"""
        hours_str = [f"{h:02d}:00" for h in self.hourly_vol.index]
        volumes = self.hourly_vol.values

        # Tạo mảng màu: Top 3 giờ màu Đỏ, các giờ còn lại màu Xanh nhạt
        colors = ['#ef5350' if h in self.top_3_hours else '#90caf9' for h in self.hourly_vol.index]

        # Tạo Text hiển thị Rank 1, 2, 3 trên cột
        texts = []
        for h in self.hourly_vol.index:
            if h in self.top_3_hours:
                rank = self.top_3_hours.index(h) + 1
                texts.append(f"Top {rank}")
            else:
                texts.append("")

        fig = go.Figure(data=[go.Bar(
            x=hours_str,
            y=volumes,
            marker_color=colors,
            text=texts,
            textposition='outside',
            textfont=dict(color='#d32f2f', size=12, family="Arial Black")
        )])

        fig.update_layout(
            title=f"EDA: Phân Phối Volume Trung Bình Theo Giờ - {symbol} (UTC)",
            xaxis_title="Thời gian (Giờ UTC)",
            yaxis_title="Volume Trung Bình / 5 Phút",
            template="plotly_white",
            height=400,
            margin=dict(l=40, r=40, t=60, b=40)
        )
        fig.update_yaxes(showgrid=True, gridcolor='#e6e6e6')
        fig.show()

    def run_analysis(self):
        trades = []

        # CHẠY BACKTEST TỪNG NGÀY
        for date, daily_df in self.df.groupby(self.df.index.date):
            if len(daily_df) < 200: continue

            daily_close_price = daily_df.iloc[-1]['Close']

            # KIỂM TRA BREAKOUT TRÊN TOP 3 KHUNG GIỜ
            for target_hour in self.top_3_hours:
                session_df = daily_df[daily_df.index.hour >= target_hour]
                if len(session_df) <= self.or_candles: continue

                # 1. Xác định hộp Đỉnh - Đáy
                or_df = session_df.iloc[:self.or_candles]
                or_high = or_df['High'].max()
                or_low = or_df['Low'].min()

                # 2. Tìm kiếm sự phá vỡ
                post_or_df = session_df.iloc[self.or_candles:]

                long_breaks = post_or_df[post_or_df['Close'] > or_high]
                short_breaks = post_or_df[post_or_df['Close'] < or_low]

                first_long_time = long_breaks.index[0] if not long_breaks.empty else pd.Timestamp.max
                first_short_time = short_breaks.index[0] if not short_breaks.empty else pd.Timestamp.max

                side = None
                entry_candle = None

                # 3. Quyết định chiều đánh dựa trên sự kiện nào đến trước
                if first_long_time < first_short_time:
                    side = 'LONG'
                    entry_candle = long_breaks.iloc[0]
                elif first_short_time < first_long_time:
                    side = 'SHORT'
                    entry_candle = short_breaks.iloc[0]

                if side:
                    entry_price = entry_candle['Close']
                    entry_time = entry_candle.name

                    # 4. Tính toán lợi nhuận
                    if side == 'LONG':
                        ret = (daily_close_price - entry_price) / entry_price
                    else: # SHORT
                        ret = (entry_price - daily_close_price) / entry_price

                    is_win = 1 if ret > 0 else 0

                    trades.append({
                        'Session_Hour': f"{target_hour:02d}:00",
                        'Side': side,
                        'Entry_Time': entry_time,
                        'Return': ret,
                        'Is_Win': is_win
                    })

        # TỔNG HỢP KẾT QUẢ
        trades_df = pd.DataFrame(trades)
        if trades_df.empty: return self.top_3_hours, pd.DataFrame()

        stats = trades_df.groupby(['Session_Hour', 'Side']).agg(
            Total_Trades=('Is_Win', 'count'),
            Wins=('Is_Win', 'sum'),
            Avg_Return=('Return', 'mean')
        ).reset_index()

        stats['Winrate_%'] = (stats['Wins'] / stats['Total_Trades']) * 100
        stats['Avg_Return_%'] = stats['Avg_Return'] * 100

        stats = stats.sort_values(by='Winrate_%', ascending=False).reset_index(drop=True)

        return self.top_3_hours, stats

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print(f"STARTING STRATEGY: {CONFIG['OR_DURATION_MINS']}-MINUTES OPENING RANGE BREAKOUT (LONG & SHORT)\n")

    for symbol in CONFIG['SYMBOLS']:
        print(f"Loading data for {symbol}...")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            analyzer = TrueORBAnalyzer(df_raw, CONFIG['OR_DURATION_MINS'])

            # --- 1. SHOW EDA PLOT ---
            if CONFIG['SHOW_EDA_PLOTS']:
                analyzer.plot_volume_eda(symbol)

            # --- 2. RUN BACKTEST ---
            top_hours, stats_df = analyzer.run_analysis()

            top_hours_str = ", ".join([f"{h:02d}:00 UTC" for h in top_hours])

            print("="*75)
            print(f" ASSET: {symbol}")
            print(f" Top 3 Volume Hours (OR Measured here): {top_hours_str}")
            print("-" * 75)

            if not stats_df.empty:
                stats_df['Winrate_%'] = stats_df['Winrate_%'].map("{:.2f}".format)
                stats_df['Avg_Return_%'] = stats_df['Avg_Return_%'].map("{:.3f}".format)
                print(" PERFORMANCE REPORT (Ranked by Winrate)")
                print("-" * 75)
                print(stats_df.to_string(index=False))
            else:
                print(" No valid breakout trades found.")
            print("="*75 + "\n")

STARTING STRATEGY: 15-MINUTES OPENING RANGE BREAKOUT (LONG & SHORT)

Loading data for BTCUSDT...


 ASSET: BTCUSDT
 Top 3 Volume Hours (OR Measured here): 15:00 UTC, 14:00 UTC, 16:00 UTC
---------------------------------------------------------------------------
 PERFORMANCE REPORT (Ranked by Winrate)
---------------------------------------------------------------------------
Session_Hour  Side  Total_Trades  Wins  Avg_Return Winrate_% Avg_Return_%
       16:00  LONG           168    86    0.001939     51.19        0.194
       14:00  LONG           193    96    0.001085     49.74        0.109
       15:00  LONG           186    89    0.002404     47.85        0.240
       15:00 SHORT           177    81   -0.000462     45.76       -0.046
       14:00 SHORT           172    77   -0.002926     44.77       -0.293
       16:00 SHORT           196    77   -0.002274     39.29       -0.227

Loading data for ETHUSDT...


 ASSET: ETHUSDT
 Top 3 Volume Hours (OR Measured here): 15:00 UTC, 14:00 UTC, 16:00 UTC
---------------------------------------------------------------------------
 PERFORMANCE REPORT (Ranked by Winrate)
---------------------------------------------------------------------------
Session_Hour  Side  Total_Trades  Wins  Avg_Return Winrate_% Avg_Return_%
       16:00  LONG           174    93    0.002217     53.45        0.222
       15:00  LONG           180    90    0.002228     50.00        0.223
       14:00 SHORT           174    86   -0.001652     49.43       -0.165
       15:00 SHORT           184    84    0.000285     45.65        0.029
       14:00  LONG           192    87    0.000854     45.31        0.085
       16:00 SHORT           190    79   -0.000871     41.58       -0.087

Loading data for SOLUSDT...


 ASSET: SOLUSDT
 Top 3 Volume Hours (OR Measured here): 15:00 UTC, 14:00 UTC, 16:00 UTC
---------------------------------------------------------------------------
 PERFORMANCE REPORT (Ranked by Winrate)
---------------------------------------------------------------------------
Session_Hour  Side  Total_Trades  Wins  Avg_Return Winrate_% Avg_Return_%
       16:00  LONG           158    85    0.006195     53.80        0.620
       15:00 SHORT           178    91   -0.001920     51.12       -0.192
       15:00  LONG           188    95    0.002502     50.53        0.250
       14:00 SHORT           188    92   -0.004596     48.94       -0.460
       14:00  LONG           178    84   -0.000030     47.19       -0.003
       16:00 SHORT           206    96   -0.001680     46.60       -0.168

Loading data for ARBUSDT...


 ASSET: ARBUSDT
 Top 3 Volume Hours (OR Measured here): 15:00 UTC, 16:00 UTC, 14:00 UTC
---------------------------------------------------------------------------
 PERFORMANCE REPORT (Ranked by Winrate)
---------------------------------------------------------------------------
Session_Hour  Side  Total_Trades  Wins  Avg_Return Winrate_% Avg_Return_%
       15:00  LONG           141    78    0.004485     55.32        0.448
       16:00  LONG           138    70    0.003330     50.72        0.333
       15:00 SHORT           143    69   -0.002495     48.25       -0.250
       14:00  LONG           142    65    0.001181     45.77        0.118
       14:00 SHORT           142    64   -0.003364     45.07       -0.336
       16:00 SHORT           146    60   -0.003536     41.10       -0.354

Loading data for BNBUSDT...


 ASSET: BNBUSDT
 Top 3 Volume Hours (OR Measured here): 15:00 UTC, 14:00 UTC, 16:00 UTC
---------------------------------------------------------------------------
 PERFORMANCE REPORT (Ranked by Winrate)
---------------------------------------------------------------------------
Session_Hour  Side  Total_Trades  Wins  Avg_Return Winrate_% Avg_Return_%
       16:00  LONG           175    84    0.001159     48.00        0.116
       15:00  LONG           189    88   -0.000165     46.56       -0.017
       14:00 SHORT           174    79   -0.000491     45.40       -0.049
       15:00 SHORT           174    77    0.000009     44.25        0.001
       14:00  LONG           192    82   -0.001218     42.71       -0.122
       16:00 SHORT           189    75    0.000054     39.68        0.005



In [3]:
!pip install pandas requests plotly -q

import requests
import time
import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go

# ==========================================
# CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'ARBUSDT', 'BNBUSDT'],
    'INTERVAL': '5m',
    'START_DATE': '01/01/2023',
    'END_DATE': '01/01/2024',
    'OR_DURATION_MINS': 15,       # 15 phút đầu phiên (3 nến M5)
    'RR_RATIO': 2.0               # Take Profit = 2 lần Stop Loss (Tỷ lệ 1:2)
}

# ==========================================
# DATA FETCHER (Giữ nguyên)
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        try: requests.get(f"{base_url}/api/v3/time", timeout=5)
        except: base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)
        end_time = int(datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59).timestamp() * 1000)

        all_klines = []
        while start_time < end_time:
            params = {"symbol": self.symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        return df.astype(float)

# ==========================================
# OPTIMIZED ORB ANALYZER (Breakout -> Retest -> Confirm)
# ==========================================
class OptimizedORBAnalyzer:
    def __init__(self, df, or_mins, rr_ratio):
        self.df = df
        self.or_candles = int(or_mins / 5)
        self.rr_ratio = rr_ratio

        # Tìm 3 giờ có volume cao nhất (Đại diện cho phiên NY/London Open)
        self.hourly_vol = self.df.groupby(self.df.index.hour)['Volume'].mean()
        self.top_3_hours = self.hourly_vol.nlargest(3).index.tolist()

    def run_analysis(self):
        trades = []

        # Duyệt qua từng ngày
        for date, daily_df in self.df.groupby(self.df.index.date):
            if len(daily_df) < 200: continue

            # Duyệt qua 3 khung giờ vàng
            for target_hour in self.top_3_hours:
                session_df = daily_df[daily_df.index.hour >= target_hour]
                if len(session_df) <= self.or_candles: continue

                # --- SETUP: Xác định OR High/Low/Mid ---
                or_df = session_df.iloc[:self.or_candles]
                or_high = or_df['High'].max()
                or_low = or_df['Low'].min()
                or_mid = (or_high + or_low) / 2

                # State Machine Variables
                state = 'WAIT_BREAK'
                trade_side = None
                pullback_extreme = None

                entry_price, sl_price, tp_price = 0, 0, 0

                # Bắt đầu quét các nến sau 15 phút đầu phiên
                post_or_df = session_df.iloc[self.or_candles:]

                for idx, row in post_or_df.iterrows():
                    c_open, c_high, c_low, c_close = row['Open'], row['High'], row['Low'], row['Close']

                    # 1. CHỜ BREAKOUT (Đóng nến qua cản)
                    if state == 'WAIT_BREAK':
                        if c_close > or_high:
                            state = 'WAIT_RETEST'
                            trade_side = 'LONG'
                            pullback_extreme = c_low # Theo dõi điểm thấp nhất của nhịp hồi
                        elif c_close < or_low:
                            state = 'WAIT_RETEST'
                            trade_side = 'SHORT'
                            pullback_extreme = c_high

                    # 2. CHỜ RETEST (Chạm lại viền hộp mà không thủng Mid-point)
                    elif state == 'WAIT_RETEST':
                        if trade_side == 'LONG':
                            if c_low < or_mid: break # Giá sập qua luôn Mid-point -> Hủy Setup (Phá vỡ giả)
                            pullback_extreme = min(pullback_extreme, c_low)

                            if c_low <= or_high: # Râu nến chạm lại OR High
                                state = 'WAIT_CONFIRM'

                        elif trade_side == 'SHORT':
                            if c_high > or_mid: break # Giá hồi quá Mid-point -> Hủy Setup
                            pullback_extreme = max(pullback_extreme, c_high)

                            if c_high >= or_low: # Râu nến chạm lại OR Low
                                state = 'WAIT_CONFIRM'

                    # 3. CHỜ XÁC NHẬN (Nến xanh/đỏ đảo chiều) -> VÀO LỆNH
                    elif state == 'WAIT_CONFIRM':
                        if trade_side == 'LONG':
                            if c_low < or_mid: break
                            pullback_extreme = min(pullback_extreme, c_low)

                            is_green = c_close > c_open
                            if is_green:
                                entry_price = c_close
                                sl_price = pullback_extreme # SL dưới đáy nhịp Retest
                                if entry_price <= sl_price: break # Lỗi Logic an toàn

                                risk = entry_price - sl_price
                                tp_price = entry_price + (risk * self.rr_ratio)
                                state = 'IN_TRADE'
                                entry_time = idx

                        elif trade_side == 'SHORT':
                            if c_high > or_mid: break
                            pullback_extreme = max(pullback_extreme, c_high)

                            is_red = c_close < c_open
                            if is_red:
                                entry_price = c_close
                                sl_price = pullback_extreme # SL trên đỉnh nhịp Retest
                                if entry_price >= sl_price: break

                                risk = sl_price - entry_price
                                tp_price = entry_price - (risk * self.rr_ratio)
                                state = 'IN_TRADE'
                                entry_time = idx

                    # 4. QUẢN LÝ LỆNH (Chờ hit TP hoặc SL)
                    elif state == 'IN_TRADE':
                        if trade_side == 'LONG':
                            if c_low <= sl_price:
                                trades.append({'Session_Hour': f"{target_hour:02d}:00", 'Side': 'LONG', 'Result': 'LOSS'})
                                break
                            elif c_high >= tp_price:
                                trades.append({'Session_Hour': f"{target_hour:02d}:00", 'Side': 'LONG', 'Result': 'WIN'})
                                break

                        elif trade_side == 'SHORT':
                            if c_high >= sl_price:
                                trades.append({'Session_Hour': f"{target_hour:02d}:00", 'Side': 'SHORT', 'Result': 'LOSS'})
                                break
                            elif c_low <= tp_price:
                                trades.append({'Session_Hour': f"{target_hour:02d}:00", 'Side': 'SHORT', 'Result': 'WIN'})
                                break

        # --- TỔNG HỢP KẾT QUẢ ---
        trades_df = pd.DataFrame(trades)
        if trades_df.empty: return self.top_3_hours, pd.DataFrame()

        trades_df['Is_Win'] = (trades_df['Result'] == 'WIN').astype(int)

        stats = trades_df.groupby(['Session_Hour', 'Side']).agg(
            Total_Setups=('Is_Win', 'count'),
            Wins=('Is_Win', 'sum')
        ).reset_index()

        stats['Winrate_%'] = (stats['Wins'] / stats['Total_Setups']) * 100
        stats = stats.sort_values(by='Winrate_%', ascending=False).reset_index(drop=True)

        return self.top_3_hours, stats

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print("="*80)
    print(" TỐI ƯU HÓA: ORB + RETEST + CONFIRMATION (R:R 1:2)")
    print(" Mục tiêu: Tránh FOMO, SL cực ngắn tại Retest Low, TP tỷ lệ 1:2")
    print("="*80 + "\n")

    for symbol in CONFIG['SYMBOLS']:
        print(f"[*] Đang tải dữ liệu {symbol}...")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            analyzer = OptimizedORBAnalyzer(df_raw, CONFIG['OR_DURATION_MINS'], CONFIG['RR_RATIO'])
            top_hours, stats_df = analyzer.run_analysis()

            top_hours_str = ", ".join([f"{h:02d}:00 UTC" for h in top_hours])

            print(f"\n---> KẾT QUẢ: {symbol}")
            print(f"Phiên giao dịch (Proxy NY Open): {top_hours_str}")

            if not stats_df.empty:
                stats_df['Winrate_%'] = stats_df['Winrate_%'].map("{:.2f}%".format)
                print(stats_df.to_string(index=False))
            else:
                print("Không có tín hiệu Retest nào hợp lệ.")
            print("-" * 80)

 TỐI ƯU HÓA: ORB + RETEST + CONFIRMATION (R:R 1:2)
 Mục tiêu: Tránh FOMO, SL cực ngắn tại Retest Low, TP tỷ lệ 1:2

[*] Đang tải dữ liệu BTCUSDT...

---> KẾT QUẢ: BTCUSDT
Phiên giao dịch (Proxy NY Open): 15:00 UTC, 14:00 UTC, 16:00 UTC
Session_Hour  Side  Total_Setups  Wins Winrate_%
       14:00 SHORT            91    39    42.86%
       15:00  LONG           104    42    40.38%
       15:00 SHORT           100    37    37.00%
       16:00  LONG            99    35    35.35%
       14:00  LONG           101    34    33.66%
       16:00 SHORT           111    28    25.23%
--------------------------------------------------------------------------------
[*] Đang tải dữ liệu ETHUSDT...

---> KẾT QUẢ: ETHUSDT
Phiên giao dịch (Proxy NY Open): 15:00 UTC, 14:00 UTC, 16:00 UTC
Session_Hour  Side  Total_Setups  Wins Winrate_%
       15:00 SHORT            99    45    45.45%
       14:00 SHORT            95    40    42.11%
       14:00  LONG           101    38    37.62%
       15:00  LONG      

In [4]:
!pip install pandas requests -q

import requests
import time
import pandas as pd
import numpy as np
from datetime import datetime

# ==========================================
# 1. CẤU HÌNH CHIẾN LƯỢC & QUẢN LÝ VỐN
# ==========================================
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'ARBUSDT', 'BNBUSDT', 'ADAUSDT'],
    'INTERVAL': '5m',
    'START_DATE': '01/01/2023',
    'END_DATE': '01/01/2024',

    # Chiến lược ORB
    'OR_DURATION_MINS': 15,       # 15 phút đầu phiên (3 nến M5)
    'RR_RATIO': 2.0,              # Tỷ lệ R:R = 1:2

    # Quản lý vốn (Mới)
    'INITIAL_CAPITAL': 10000.0,   # Vốn khởi điểm: $10,000
    'POSITION_SIZE_PCT': 0.10,    # Đánh 10% vốn mỗi lệnh
    'FEE_RATE': 0.0004            # Phí sàn Binance (0.04%)
}

# ==========================================
# 2. MODULE TẢI DỮ LIỆU
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        return int(datetime.strptime(date_str, "%d/%m/%Y").timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        try: requests.get(f"{base_url}/api/v3/time", timeout=5)
        except: base_url = "https://api.binance.us"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)
        end_time = int(datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59).timestamp() * 1000)

        all_klines = []
        while start_time < end_time:
            params = {"symbol": self.symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        return df.astype(float)

# ==========================================
# 3. ENGINE BACKTEST & TÍNH PNL
# ==========================================
class ORBBacktestEngine:
    def __init__(self, df, config):
        self.df = df
        self.config = config
        self.or_candles = int(config['OR_DURATION_MINS'] / 5)

        # Tìm 3 giờ vàng
        self.hourly_vol = self.df.groupby(self.df.index.hour)['Volume'].mean()
        self.top_3_hours = self.hourly_vol.nlargest(3).index.tolist()

    def run(self):
        trades = []
        capital = self.config['INITIAL_CAPITAL']
        fee = self.config['FEE_RATE']

        for date, daily_df in self.df.groupby(self.df.index.date):
            if len(daily_df) < 200: continue

            for target_hour in self.top_3_hours:
                session_df = daily_df[daily_df.index.hour >= target_hour]
                if len(session_df) <= self.or_candles: continue

                or_df = session_df.iloc[:self.or_candles]
                or_high = or_df['High'].max()
                or_low = or_df['Low'].min()
                or_mid = (or_high + or_low) / 2

                state = 'WAIT_BREAK'
                trade_side = None
                pullback_extreme = None
                entry_price, sl_price, tp_price = 0, 0, 0

                post_or_df = session_df.iloc[self.or_candles:]

                for idx, row in post_or_df.iterrows():
                    c_open, c_high, c_low, c_close = row['Open'], row['High'], row['Low'], row['Close']

                    if state == 'WAIT_BREAK':
                        if c_close > or_high:
                            state = 'WAIT_RETEST'
                            trade_side = 'LONG'
                            pullback_extreme = c_low
                        elif c_close < or_low:
                            state = 'WAIT_RETEST'
                            trade_side = 'SHORT'
                            pullback_extreme = c_high

                    elif state == 'WAIT_RETEST':
                        if trade_side == 'LONG':
                            if c_low < or_mid: break
                            pullback_extreme = min(pullback_extreme, c_low)
                            if c_low <= or_high: state = 'WAIT_CONFIRM'
                        elif trade_side == 'SHORT':
                            if c_high > or_mid: break
                            pullback_extreme = max(pullback_extreme, c_high)
                            if c_high >= or_low: state = 'WAIT_CONFIRM'

                    elif state == 'WAIT_CONFIRM':
                        if trade_side == 'LONG':
                            if c_low < or_mid: break
                            pullback_extreme = min(pullback_extreme, c_low)
                            if c_close > c_open: # Nến xanh
                                entry_price = c_close
                                sl_price = pullback_extreme
                                if entry_price <= sl_price: break
                                tp_price = entry_price + ((entry_price - sl_price) * self.config['RR_RATIO'])
                                state = 'IN_TRADE'

                        elif trade_side == 'SHORT':
                            if c_high > or_mid: break
                            pullback_extreme = max(pullback_extreme, c_high)
                            if c_close < c_open: # Nến đỏ
                                entry_price = c_close
                                sl_price = pullback_extreme
                                if entry_price >= sl_price: break
                                tp_price = entry_price - ((sl_price - entry_price) * self.config['RR_RATIO'])
                                state = 'IN_TRADE'

                    elif state == 'IN_TRADE':
                        exit_price = 0
                        # Check hit SL / TP
                        if trade_side == 'LONG':
                            if c_low <= sl_price: exit_price = sl_price
                            elif c_high >= tp_price: exit_price = tp_price
                        elif trade_side == 'SHORT':
                            if c_high >= sl_price: exit_price = sl_price
                            elif c_low <= tp_price: exit_price = tp_price

                        # Tính tiền nếu lệnh đóng
                        if exit_price != 0:
                            pos_size = capital * self.config['POSITION_SIZE_PCT']

                            if trade_side == 'LONG':
                                pnl_pct = (exit_price - entry_price) / entry_price
                            else:
                                pnl_pct = (entry_price - exit_price) / entry_price

                            net_pnl_pct = pnl_pct - (fee * 2)
                            pnl_amt = pos_size * net_pnl_pct
                            capital += pnl_amt

                            trades.append({
                                'Exit_Time': idx,
                                'PnL_Amt': pnl_amt,
                                'Capital': capital
                            })
                            break # Đóng lệnh, xong phiên này
        return pd.DataFrame(trades)

# ==========================================
# 4. MODULE TỔNG HỢP (EVALUATOR)
# ==========================================
def calculate_metrics(trades_df, symbol, config):
    if trades_df.empty: return None

    initial_cap = config['INITIAL_CAPITAL']
    final_cap = trades_df.iloc[-1]['Capital']
    net_pnl = final_cap - initial_cap
    total_return = net_pnl / initial_cap

    wins = trades_df[trades_df['PnL_Amt'] > 0]
    losses = trades_df[trades_df['PnL_Amt'] <= 0]

    trades_count = len(trades_df)
    winrate = len(wins) / trades_count if trades_count > 0 else 0

    gross_profit = wins['PnL_Amt'].sum()
    gross_loss = abs(losses['PnL_Amt'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss != 0 else (99.99 if gross_profit > 0 else 0)

    trades_df['Peak'] = trades_df['Capital'].cummax()
    trades_df['DD'] = (trades_df['Capital'] - trades_df['Peak']) / trades_df['Peak']
    max_dd = trades_df['DD'].min()

    # Tính Sharpe (Chuyển lệnh sang dạng Daily Returns để tính)
    trades_df.set_index('Exit_Time', inplace=True)
    daily_pnl = trades_df.resample('D')['PnL_Amt'].sum()
    daily_capital = initial_cap + daily_pnl.cumsum()
    daily_returns = daily_capital.pct_change().dropna()

    sharpe = 0
    if daily_returns.std() != 0:
        sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(365)

    return {
        'Symbol': symbol,
        'Timeframe': '5m ORB',
        'Trades': trades_count,
        'Winrate': f"{winrate*100:.2f}%",
        'Profit_Factor': f"{profit_factor:.2f}",
        'Max_DD': f"{max_dd*100:.2f}%",
        'Net_PnL_$': f"{net_pnl:.2f}",
        'Return': f"{total_return*100:.2f}%",
        'Sharpe': f"{sharpe:.2f}"
    }

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print("Fetching data and simulating strategy...\n")
    all_results = []

    for symbol in CONFIG['SYMBOLS']:
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            engine = ORBBacktestEngine(df_raw, CONFIG)
            trades_df = engine.run()
            metrics = calculate_metrics(trades_df, symbol, CONFIG)
            if metrics:
                all_results.append(metrics)

    # Hiển thị bảng chuẩn hóa
    if all_results:
        final_df = pd.DataFrame(all_results)
        # Sắp xếp theo thứ tự Sharpe từ cao xuống thấp
        final_df['Sharpe_Num'] = final_df['Sharpe'].astype(float)
        final_df = final_df.sort_values('Sharpe_Num', ascending=False).drop('Sharpe_Num', axis=1)

        print(final_df.to_string(index=False))
    else:
        print("Không có tín hiệu giao dịch nào được tìm thấy.")

Fetching data and simulating strategy...

 Symbol Timeframe  Trades Winrate Profit_Factor Max_DD Net_PnL_$ Return Sharpe
SOLUSDT    5m ORB     632  32.44%          0.82 -5.68%   -483.62 -4.84%  -2.28
ADAUSDT    5m ORB     650  34.31%          0.74 -5.18%   -473.23 -4.73%  -2.74
ETHUSDT    5m ORB     601  35.44%          0.73 -3.46%   -339.60 -3.40%  -3.08
ARBUSDT    5m ORB     503  29.62%          0.73 -7.05%   -502.55 -5.03%  -3.58
BTCUSDT    5m ORB     606  32.67%          0.67 -4.09%   -394.27 -3.94%  -3.89
BNBUSDT    5m ORB     676  29.73%          0.52 -7.75%   -772.68 -7.73%  -6.13
